# Single-slice DLPFC

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import matplotlib.pyplot as plt

import SpaDiff as sd
from SpaDiff.utils import mclust_R, set_seed

## Configuration

In [ ]:
SEED = 42
SAMPLE_ID = "151672"
N_CLUSTERS = 5

DATA_ROOT = Path("path/DLPFC")
print("DATA_ROOT =", DATA_ROOT)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device =", device)

## Data loading

In [ ]:
sample_dir = DATA_ROOT / SAMPLE_ID
adata = sc.read_visium(sample_dir)
adata.var_names_make_unique()

adata.layers["counts"] = adata.X.copy()  
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=3000, subset=True)
# sc.pp.scale(adata)
truth = pd.read_csv(sample_dir / "truth.txt", sep="\t", header=None, index_col=0)
truth.columns = ["Truth"]
adata.obs["Truth"] = truth.reindex(adata.obs_names)["Truth"]
adata

## Simplicial complex

In [ ]:
# adata, adjacency = sd.spatial_reconstruction(adata, alpha=2.0) 
# operators = sd.to_torch_operators(sd.build_simplicial_operators(adjacency), device=device)
topology = sd.build_spatial_topology(adata, mode="global_knn", device=device)
operators = topology.operators

sc.tl.pca(adata, n_comps=50)
features = torch.as_tensor(np.asarray(adata.obsm["X_pca"]), dtype=torch.float32, device=device)

In [ ]:
print("features:", tuple(features.shape))

## Conditional VP-SDE training

In [ ]:
config = sd.SpaDiffConfig(
    num_batches=1,
)
model = sd.SpaDiff(config).to(device)

adata = model.fit_transform(
    adata,
    features,
    operators,
    batch_key=None,
    ode_steps=200,
)

## Spatial domains and ARI

In [ ]:
labels = mclust_R( adata, num_cluster=N_CLUSTERS, used_obsm="spadiff", pca_num=20, random_seed=SEED )
adata.obs["mclust"] = pd.Categorical(labels.astype(str))
valid = adata.obs[["mclust", "Truth"]].dropna()
ari = round(adjusted_rand_score(valid["Truth"], valid["mclust"]), 3)
nmi = normalized_mutual_info_score(valid["Truth"], valid["mclust"])

print(f"ARI = {ari:.3f}")
print(f"NMI = {nmi:.3f}")

palette = ["#6D1A9C", "#D1D1D1", "#F56867", "#59BE86", "#FEB915", "#C798EE", "#7495D3"]
sc.pl.spatial(
    adata, img_key="hires", color="mclust", palette=palette,
    title=f"SpaDiff | ARI={ari:.3f}", legend_loc=None,
    frameon=False, spot_size=120,
    show=False,
)
# plt.savefig("../result/spadiff_"+SAMPLE_ID+"_"+str(ari)+".pdf", bbox_inches='tight')

In [ ]:
# output_file ="../result/spadiff_"+SAMPLE_ID+"_"+str(ari)+".h5ad"  # Compressed output path

# adata.write_h5ad(output_file, compression="gzip")